# Chapter 03 — Real-Time Streaming, Windowing & Late Data

Welcome to the interactive notebook for **Chapter 03**!

> 📖 **Concept Reference**: For in-depth theory on event-time semantics, watermarks, triggers, and allowed lateness, see [Chapter 03 README](README.md#core-streaming-concepts-the-4-questions).

### Interactive Workflow
1. Inspect synthetic telemetry records with simulated network lag and clock drift.
2. Parse timestamps and assign event time using `beam.window.TimestampedValue`.
3. Execute 3 parallel windowing topologies: **Fixed (Tumbling)**, **Sliding (Hopping)**, and **Session Windows**.
4. Inspect output files and late-data trigger emissions.

## 1. Environment Setup & Kernel Selection

This project uses **[`uv`](https://docs.astral.sh/uv/)** to manage dependencies in an isolated `.venv` virtual environment.

> 📘 **Full Environment Setup Guide**: For complete setup instructions, terminal commands, and kernel registration, see the root [README.md](../README.md#environment-setup--activating-the-kernel-with-uv).

* **Option A: In the Terminal**: Run `source .venv/bin/activate` or execute commands directly with `uv run`.
* **Option B: In This Notebook (Kernel Selection)**: Click the **Kernel Picker** in the top-right corner and select **`Python (dataflow-workshops)`** (or browse to `.venv/bin/python`).

Run the verification cell below to confirm your setup:


In [1]:
import os
import sys

print(f"Python Executable: {sys.executable}")
if ".venv" in sys.executable:
    print("✅ Success: Notebook is running inside the project's .venv!")
else:
    print("⚠️ Warning: Notebook is running on system Python! Select 'Python (dataflow-workshops)' from the top-right kernel picker.")

import apache_beam as beam
print(f"✅ Apache Beam Version: {beam.__version__}")

Python Executable: /Users/zelda.luconi/Medium/google-products/dataflow-workshops/.venv/bin/python3
✅ Success: Notebook is running inside the project's .venv!


✅ Apache Beam Version: 2.77.0


## 2. Streaming Architecture & Visual DAG

Here is the architectural data flow for this chapter. Telemetry records are ingested, validated, and fanned out into three parallel window branches:

![Chapter 03 Streaming DAG](../artifacts/03_windowing_and_streaming_dag.png)

<details>
<summary><b>Click to expand Mermaid source code</b></summary>

```mermaid
flowchart LR
    classDef pcoll fill:#e8f0fe,stroke:#4285f4,stroke-width:2px,color:#1a73e8,font-weight:bold;
    classDef ptrans fill:#fef7e0,stroke:#f9ab00,stroke-width:2px,color:#b06000,font-weight:bold;
    classDef sink fill:#e6f4ea,stroke:#34a853,stroke-width:2px,color:#137333,font-weight:bold;
    classDef dlq fill:#fce8e6,stroke:#ea4335,stroke-width:2px,color:#c5221f,font-weight:bold;
    classDef win fill:#f3e8fd,stroke:#9334e6,stroke-width:2px,color:#6b21a8,font-weight:bold;

    subgraph StreamIngest["1. Ingestion & Validation"]
        PUBSUB[("Pub/Sub Topic<br/>telemetry-events")]:::pcoll
        READ["'ReadFromPubSub'"]:::ptrans
        PARSE["'ParseAndValidate'<br/><i>TimestampedValue</i>"]:::ptrans
        VALID[("Valid Stream")]:::pcoll
        DLQ_RECS[("DLQ Records")]:::dlq
        DLQ_SINK[("Dead-Letter Sink")]:::dlq
        PUBSUB --> READ --> PARSE
        PARSE --> VALID
        PARSE -- "Malformed" --> DLQ_RECS --> DLQ_SINK
    end

    subgraph WindowingBranches["2. Temporal Windowing Strategies & Sinks"]
        direction TB
        subgraph BranchA["Branch A: Tumbling Windows (60s)"]
            WIN_TUMB["FixedWindows(60s)<br/><i>Trigger + Allowed Lateness</i>"]:::win
            COMB_TUMB["CombinePerKey<br/>'CombineTumbling'"]:::ptrans
            BQ_TUMB[("BigQuery: kpi_tumbling")]:::sink
            WIN_TUMB --> COMB_TUMB --> BQ_TUMB
        end

        subgraph BranchB["Branch B: Sliding Windows (60s, period=20s)"]
            WIN_SLIDE["SlidingWindows(60s, 20s)<br/><i>Rolling Trend Rate</i>"]:::win
            COMB_SLIDE["CombinePerKey<br/>'CombineSliding'"]:::ptrans
            BQ_SLIDE[("BigQuery: kpi_sliding")]:::sink
            WIN_SLIDE --> COMB_SLIDE --> BQ_SLIDE
        end

        subgraph BranchC["Branch C: Session Windows (30s Gap)"]
            WIN_SESS["Sessions(30s)<br/><i>Activity Bursts</i>"]:::win
            COMB_SESS["GroupByKey + Aggregate<br/>'AggregateSession'"]:::ptrans
            BQ_SESS[("BigQuery: kpi_session")]:::sink
            WIN_SESS --> COMB_SESS --> BQ_SESS
        end
    end

    VALID --> WIN_TUMB
    VALID --> WIN_SLIDE
    VALID --> WIN_SESS
```
</details>

## 3. Inspecting Synthetic Streaming Telemetry

Let's inspect our synthetic streaming dataset (`data/sample_events.json`).
Notice the following deliberate scenarios:
1. **On-time events** with `event_timestamp` in ISO-8601.
2. **Simulated late arrival** (`evt-1013`, timestamped at `10:00:30` but emitted out-of-order).
3. **Poison records**: `evt-1014` (negative amount), `evt-1015` (empty user_id), and a malformed raw JSON string.

In [2]:
from pathlib import Path
import json

raw_text = Path("data/sample_events.json").read_text()
print(f"Total Characters: {len(raw_text)}")
print("Sample lines from data/sample_events.json:")
for line in raw_text.splitlines()[:15]:
    print(line)

Total Characters: 2881
Sample lines from data/sample_events.json:
[
  {"event_id": "evt-1001", "user_id": "usr_alpha", "device_id": "dev_01", "event_type": "page_view", "page": "/home", "amount": 0.0, "event_timestamp": "2026-10-27T10:00:05Z"},
  {"event_id": "evt-1002", "user_id": "usr_alpha", "device_id": "dev_01", "event_type": "view_item", "page": "/product/101", "amount": 0.0, "event_timestamp": "2026-10-27T10:00:15Z"},
  {"event_id": "evt-1003", "user_id": "usr_beta", "device_id": "dev_02", "event_type": "page_view", "page": "/home", "amount": 0.0, "event_timestamp": "2026-10-27T10:00:20Z"},
  {"event_id": "evt-1004", "user_id": "usr_alpha", "device_id": "dev_01", "event_type": "add_to_cart", "page": "/cart", "amount": 89.99, "event_timestamp": "2026-10-27T10:00:35Z"},
  {"event_id": "evt-1005", "user_id": "usr_gamma", "device_id": "dev_03", "event_type": "page_view", "page": "/categories", "amount": 0.0, "event_timestamp": "2026-10-27T10:00:40Z"},
  {"event_id": "evt-1006", "use

## 4. Event Time & Watermark Assignment

We extract event-time timestamps from payloads and wrap elements with `TimestampedValue`.
*(For the theoretical distinction between event time, processing time, and watermarks, see [Chapter 03 README](README.md#event-time-vs-processing-time)).*

In [3]:
import datetime
import apache_beam as beam

sample_records = [
    {"user": "alice", "spend": 25.0, "ts": "2026-10-27T10:00:10Z"},
    {"user": "bob",   "spend": 50.0, "ts": "2026-10-27T10:00:25Z"},
    {"user": "alice", "spend": 75.0, "ts": "2026-10-27T10:00:50Z"},
]

class AttachTimestampFn(beam.DoFn):
    def process(self, element):
        dt = datetime.datetime.fromisoformat(element["ts"].replace("Z", "+00:00"))
        yield beam.window.TimestampedValue(element, dt.timestamp())

class InspectTimestampFn(beam.DoFn):
    def process(self, element, ts=beam.DoFn.TimestampParam):
        dt_str = datetime.datetime.fromtimestamp(float(ts), tz=datetime.timezone.utc).isoformat()
        print(f"Element: {element['user']:<6} | Spend: ${element['spend']:<5.1f} | Assigned Event Time: {dt_str}")

with beam.Pipeline() as p:
    (
        p
        | "CreateEvents" >> beam.Create(sample_records)
        | "AttachTimestamp" >> beam.ParDo(AttachTimestampFn())
        | "InspectTimestamps" >> beam.ParDo(InspectTimestampFn())
    )

Element: alice  | Spend: $25.0  | Assigned Event Time: 2026-10-27T10:00:10+00:00
Element: bob    | Spend: $50.0  | Assigned Event Time: 2026-10-27T10:00:25+00:00
Element: alice  | Spend: $75.0  | Assigned Event Time: 2026-10-27T10:00:50+00:00


## 5. Branch A: Fixed (Tumbling) Windows & Triggers

Aggregates metrics in non-overlapping 60s windows with early and late watermark triggers.
*(For trigger accumulation modes and late data policies, see [Chapter 03 README](README.md#1-fixed-tumbling-windows)).*

In [4]:
from apache_beam.transforms import window

class FormatWindowSummaryFn(beam.DoFn):
    def process(self, element, win=beam.DoFn.WindowParam):
        user, total = element
        start = win.start.to_utc_datetime().strftime("%H:%M:%S")
        end = win.end.to_utc_datetime().strftime("%H:%M:%S")
        print(f"Window [{start} -> {end}] | User: {user:<6} | Total Spend: ${total:.2f}")

with beam.Pipeline() as p:
    (
        p
        | "CreateEvents" >> beam.Create(sample_records)
        | "AttachTimestamp" >> beam.ParDo(AttachTimestampFn())
        | "Fixed60s" >> beam.WindowInto(window.FixedWindows(60))
        | "KeyByUser" >> beam.Map(lambda x: (x["user"], x["spend"]))
        | "SumSpend" >> beam.CombinePerKey(sum)
        | "FormatSummary" >> beam.ParDo(FormatWindowSummaryFn())
    )

Window [10:00:00 -> 10:01:00] | User: alice  | Total Spend: $100.00
Window [10:00:00 -> 10:01:00] | User: bob    | Total Spend: $50.00


## 6. Branch B: Sliding (Hopping) Windows

Computes rolling 60s velocity metrics every 20s for moving trends.
*(For sliding window mechanics and state reuse, see [Chapter 03 README](README.md#2-sliding-hopping-windows)).*

In [5]:
with beam.Pipeline() as p:
    (
        p
        | "CreateEvents" >> beam.Create(sample_records)
        | "AttachTimestamp" >> beam.ParDo(AttachTimestampFn())
        | "SlidingWindow" >> beam.WindowInto(window.SlidingWindows(size=60, period=20))
        | "KeyByUser" >> beam.Map(lambda x: (x["user"], x["spend"]))
        | "SumSpend" >> beam.CombinePerKey(sum)
        | "FormatSliding" >> beam.ParDo(FormatWindowSummaryFn())
    )

Window [10:00:00 -> 10:01:00] | User: alice  | Total Spend: $100.00
Window [09:59:40 -> 10:00:40] | User: alice  | Total Spend: $25.00
Window [09:59:20 -> 10:00:20] | User: alice  | Total Spend: $25.00
Window [10:00:40 -> 10:01:40] | User: alice  | Total Spend: $75.00
Window [10:00:20 -> 10:01:20] | User: alice  | Total Spend: $75.00
Window [10:00:20 -> 10:01:20] | User: bob    | Total Spend: $50.00
Window [10:00:00 -> 10:01:00] | User: bob    | Total Spend: $50.00
Window [09:59:40 -> 10:00:40] | User: bob    | Total Spend: $50.00


## 7. Branch C: Session Windows (Activity Gaps)

Dynamically groups bursts of user activity separated by a 30s inactivity gap.
*(For session window merging rules, see [Chapter 03 README](README.md#3-session-windows)).*

In [6]:
session_data = [
    {"user": "alice", "spend": 10.0, "ts": "2026-10-27T10:00:00Z"},
    {"user": "alice", "spend": 20.0, "ts": "2026-10-27T10:00:15Z"},  # 15s gap (< 30s) -> merged!
    {"user": "alice", "spend": 30.0, "ts": "2026-10-27T10:00:35Z"},  # 20s gap (< 30s) -> merged!
    {"user": "alice", "spend": 99.0, "ts": "2026-10-27T10:05:00Z"},  # > 4min gap (> 30s) -> NEW SESSION!
]

class FormatSessionFn(beam.DoFn):
    def process(self, element, win=beam.DoFn.WindowParam):
        user, total = element
        duration = (win.end.micros - win.start.micros) / 1_000_000.0
        start = win.start.to_utc_datetime().strftime("%H:%M:%S")
        end = win.end.to_utc_datetime().strftime("%H:%M:%S")
        print(f"Session [{start} -> {end}] (Duration: {duration:4.1f}s) | User: {user:<6} | Total Spend: ${total:.2f}")

with beam.Pipeline() as p:
    (
        p
        | "CreateSessionEvents" >> beam.Create(session_data)
        | "AttachTimestamp" >> beam.ParDo(AttachTimestampFn())
        | "Session30s" >> beam.WindowInto(window.Sessions(gap_size=30))
        | "KeyByUser" >> beam.Map(lambda x: (x["user"], x["spend"]))
        | "SumSessionSpend" >> beam.CombinePerKey(sum)
        | "FormatSession" >> beam.ParDo(FormatSessionFn())
    )

Session [10:00:00 -> 10:01:05] (Duration: 65.0s) | User: alice  | Total Spend: $60.00
Session [10:05:00 -> 10:05:30] (Duration: 30.0s) | User: alice  | Total Spend: $99.00


## 8. Running the Complete Production Streaming Pipeline Locally

Now let's execute the complete pipeline script [`pipelines/streaming_windowing_pipeline.py`](pipelines/streaming_windowing_pipeline.py) using `DirectRunner`.

This will process the entire `data/sample_events.json` dataset across all 3 windowing branches and output the results to `output/`:

In [7]:
import subprocess

cmd = ["bash", "scripts/01_run_local.sh"]
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)

 Chapter 03: Running Streaming Windowing & Late Data Pipeline Locally
Input Events:     /Users/zelda.luconi/Medium/google-products/dataflow-workshops/03_realtime_streaming/data/sample_events.json
Tumbling Output:  /Users/zelda.luconi/Medium/google-products/dataflow-workshops/03_realtime_streaming/output/tumbling_metrics*.json
Sliding Output:   /Users/zelda.luconi/Medium/google-products/dataflow-workshops/03_realtime_streaming/output/sliding_metrics*.json
Session Output:   /Users/zelda.luconi/Medium/google-products/dataflow-workshops/03_realtime_streaming/output/session_metrics*.json
Dead-Letter DLQ:  /Users/zelda.luconi/Medium/google-products/dataflow-workshops/03_realtime_streaming/output/dead_letters*.json
----------------------------------------------------------------------
----------------------------------------------------------------------
 Execution finished! Output files in /Users/zelda.luconi/Medium/google-products/dataflow-workshops/03_realtime_streaming/output:
total 32
dr

## 9. Inspecting Local Output Files

Let's inspect the JSON output generated for each branch in the `output/` directory:

In [8]:
import glob
import json

def show_json_lines(pattern, title):
    print(f"=== {title} ===")
    files = glob.glob(pattern)
    if not files:
        print("No files found for pattern:", pattern)
        return
    for f in sorted(files):
        with open(f, "r") as fp:
            for line in fp.readlines()[:4]:
                item = json.loads(line.strip())
                print(json.dumps(item, indent=2))
    print()

show_json_lines("output/tumbling_metrics*.json", "Fixed (Tumbling) Windows")
show_json_lines("output/sliding_metrics*.json", "Sliding (Hopping) Windows")
show_json_lines("output/session_metrics*.json", "Session Windows")
show_json_lines("output/dead_letters*.json", "Dead-Letter Poison Records")

=== Fixed (Tumbling) Windows ===
{
  "window_start": "2026-10-27T10:00:00",
  "window_end": "2026-10-27T10:01:00",
  "user_id": "usr_alpha",
  "event_count": 4,
  "total_spend": 179.98,
  "emitted_at": "2026-10-27T18:17:00.094684+00:00"
}
{
  "window_start": "2026-10-27T10:00:00",
  "window_end": "2026-10-27T10:01:00",
  "user_id": "usr_alpha",
  "event_count": 4,
  "total_spend": 179.98,
  "emitted_at": "2026-10-27T18:17:00.095503+00:00"
}
{
  "window_start": "2026-10-27T10:05:00",
  "window_end": "2026-10-27T10:06:00",
  "user_id": "usr_alpha",
  "event_count": 2,
  "total_spend": 29.99,
  "emitted_at": "2026-10-27T18:17:00.095538+00:00"
}
{
  "window_start": "2026-10-27T10:00:00",
  "window_end": "2026-10-27T10:01:00",
  "user_id": "usr_beta",
  "event_count": 3,
  "total_spend": 149.5,
  "emitted_at": "2026-10-27T18:17:00.095567+00:00"
}

=== Sliding (Hopping) Windows ===
{
  "window_start": "2026-10-27T10:00:00",
  "window_end": "2026-10-27T10:01:00",
  "user_id": "usr_alpha",
  "

## 10. Summary & Next Steps

### What You Learned in Chapter 03:
1. **Event Time vs Processing Time**: Using `TimestampedValue` to guarantee deterministic analytics regardless of arrival delays.
2. **Fixed Windows**: Creating rigid periodic summaries (e.g. 1-minute financial intervals).
3. **Sliding Windows**: Constructing overlapping intervals for moving averages and real-time velocity calculations.
4. **Session Windows**: Dynamic data-driven windows that automatically merge bursts of user activity.
5. **Dead-Letter Queues**: Ensuring poison messages are isolated into separate sinks without crashing 24/7 streaming jobs.

### Deploying to Google Cloud Dataflow:
When you are ready to deploy to GCP with **Streaming Engine** and **BigQuery Storage Write API**, run:

```bash
bash scripts/02_run_dataflow.sh
```

In **Chapter 04**, we will explore **Stateful Processing, Timers & Hot Key Fanout** (`@StateId`, `@TimerId`, and `withHotKeyFanout`).